# AdaRetrieve: source-only admission, retrieval depth and truncation

Select a T4 runtime. Upload the V5 project and index ZIP files to the session, then run all cells. The embedded runner is identical to sensitivity_run.py. The final cell exports the raw record, source profiles, freeze hashes and hardware information.

In [ ]:
import os,sys, tempfile,zipfile,hashlib,json,importlib.util,subprocess
from pathlib import Path
from google.colab import files
import torch
assert torch.cuda.is_available() and 'T4' in torch.cuda.get_device_name(), 'Select a Tesla T4 and reconnect.'
content=Path('/content')
expected={'AdaRetrieve-project.zip':'3409513b06232df6c6bbab7bb11903bec85522af17f80fc039a09db833409b2c','AdaRetrieve-indexes-and-scores.zip':'3af2b67a10413a7b6e7de78aea782d1335cb999af380c8b0bb00f89a20176340'}
if not all((content/n).is_file() for n in expected):
    print('Upload both original ZIP files to the runtime.')
    files.upload()
for n,h in expected.items():
    assert hashlib.sha256((content/n).read_bytes()).hexdigest()==h,n
work=Path(tempfile.mkdtemp(prefix='sensitivity-',dir=content))
with zipfile.ZipFile(content/'AdaRetrieve-project.zip') as z:
    for n in z.namelist():
        if n.startswith(('project/','routing-followup/')): z.extract(n,work)
with zipfile.ZipFile(content/'AdaRetrieve-indexes-and-scores.zip') as z:
    for n in z.namelist():
        if n.endswith(('.bm.pkl','.faiss')): z.extract(n,work/'project/cache')
if importlib.util.find_spec('faiss') is None:
    subprocess.run([sys.executable,'-m','pip','install','-q','faiss-cpu'],check=True)
print('VERIFIED',torch.cuda.get_device_name(),work,flush=True)

In [ ]:
runner_source = (
    '"""Source-only latency admission, held-out candidate-depth and truncation diagnostics."""\n'
    'import hashlib\n'
    'import json\n'
    'import random\n'
    'import sys\n'
    'from importlib.metadata import version\n'
    'from pathlib import Path\n'
    'from time import perf_counter_ns\n'
    '\n'
    'import faiss\n'
    'import numpy as np\n'
    'import torch\n'
    'import torch.nn.functional as F\n'
    '\n'
    'BASE = Path(__file__).resolve().parent\n'
    'sys.path.insert(0, str(BASE / "routing-followup"))\n'
    'from experiment import FEATURES, ReuseEngine, Revision, Router, State\n'
    'from neural import CE, ENC, Models\n'
    'from policy import ACTIONS, LAMBDA\n'
    'from retrieval import Collection, DOMAINS, fused, metrics, order\n'
    'from study import Engine\n'
    '\n'
    'OUT = BASE / "sensitivity-output"\n'
    'PROJECT = BASE / "project"\n'
    'FOLLOW = BASE / "routing-followup/output"\n'
    'QUANTILES = {"p50": .50, "p90": .90, "p95": .95}\n'
    'CAP = 384\n'
    'SAMPLES = 96\n'
    '\n'
    '\n'
    'def ms(start):\n'
    '    return (perf_counter_ns() - start) / 1e6\n'
    '\n'
    '\n'
    'def read(path):\n'
    '    return json.loads(path.read_text())\n'
    '\n'
    '\n'
    'def digest(path):\n'
    '    return hashlib.sha256(path.read_bytes()).hexdigest()\n'
    '\n'
    '\n'
    'def save(name, obj):\n'
    '    with (OUT / name).open("x") as file:\n'
    '        json.dump(obj, file, separators=(",", ":"), allow_nan=False)\n'
    '\n'
    '\n'
    'def log(*args):\n'
    '    print(*args, flush=True)\n'
    '\n'
    '\n'
    'def profiles(engine):\n'
    '    ds = engine.ds\n'
    '    reuse = ReuseEngine(engine)\n'
    '    ids = order(ds.source_ids()[0], "admission-profile-v1")\n'
    '    assert len(ids) >= SAMPLES + 6\n'
    '    rows = {}\n'
    '    for idx, q in enumerate(ids[:SAMPLES + 6]):\n'
    '        text = ds.queries[q]\n'
    '        start = perf_counter_ns()\n'
    '        get, score = reuse.operations(text)\n'
    '        state = State(text, get("initial"))\n'
    '        times = {"initial": ms(start)}\n'
    '        for action in ACTIONS[1:]:\n'
    '            start = perf_counter_ns()\n'
    '            state = state.next(action, get, score)\n'
    '            times[action] = ms(start)\n'
    '        if idx >= 6:\n'
    '            rows[q] = times\n'
    '    return rows\n'
    '\n'
    '\n'
    'def freeze(engines):\n'
    '    samples = {d: profiles(engines[d]) for d in DOMAINS}\n'
    '    save("source-observations.json", samples)\n'
    '    gate = read(FOLLOW / "source-gate.json")\n'
    '    seals = {}\n'
    '    for d in DOMAINS:\n'
    '        file = FOLLOW / f"frozen-{d}.json"\n'
    '        fold = read(file)\n'
    '        rootfile = PROJECT / "results" / f"{d}.router.json"\n'
    '        assert digest(file) == gate["frozen_hashes"][d] and digest(rootfile) == fold["root_sha256"]\n'
    '        sources = sorted(set(DOMAINS) - {d})\n'
    '        assert sorted(fold["sources"]) == sources\n'
    '        pooled = [r for source in sources for r in samples[source].values()]\n'
    '        table = {name: {a: float(np.quantile([r[a] for r in pooled], prob))\n'
    '                        for a in pooled[0]} for name, prob in QUANTILES.items()}\n'
    '        table["v5"] = fold["profile"]\n'
    '        name = f"admission-{d}.json"\n'
    '        save(name, dict(target=d, sources=sources, samples_per_source=SAMPLES,\n'
    '                        budgets=fold["budgets"], profiles=table, quantile_method="linear",\n'
    '                        thresholds=fold["calibration"], frozen_v5=digest(file),\n'
    '                        frozen_root=digest(rootfile), runner=digest(Path(__file__))))\n'
    '        seals[d] = digest(OUT / name)\n'
    '    save("freeze-checksums.json", seals)\n'
    '    log("FROZEN before opening test qrels", seals)\n'
    '    return seals, samples\n'
    '\n'
    '\n'
    'def costs(paths, profile):\n'
    '    return np.array([sum(profile[a] for a in path) for path in paths])\n'
    '\n'
    '\n'
    'class Admission:\n'
    '    def __init__(self, fold, spec, guard):\n'
    '        base = fold["profile"]\n'
    '        self.root = Router({**spec, "cost": costs(spec["paths"], base)})\n'
    '        self.revision = Revision(fold["models"], base)\n'
    '        self.guard = guard\n'
    '        self.root_times = costs(self.root.paths, guard)\n'
    '        self.suffix_times = {prefix: costs(model[0], guard)\n'
    '                             for prefix, model in self.revision.models.items()}\n'
    '\n'
    '    def run(self, reuse, text, budget, threshold):\n'
    '        start = perf_counter_ns()\n'
    '        get, score = reuse.operations(text)\n'
    '        s = State(text, get("initial"))\n'
    '        initial_ms = ms(start)\n'
    '        stage = {"initial": initial_ms}\n'
    '        then = perf_counter_ns()\n'
    '        # Feature elapsed is real. Only the original admission mask is disabled here.\n'
    '        pred = self.root.scores(s.features(budget, initial_ms), budget, -np.inf)\n'
    '        pred[(self.root_times > 0) & (self.root_times + initial_ms > budget)] = -np.inf\n'
    '        remaining = tuple(self.root.paths[int(pred.argmax())])\n'
    '        stage["root"] = ms(then)\n'
    '        while remaining:\n'
    '            action, remaining = remaining[0], remaining[1:]\n'
    '            if action not in s.legal() or ms(start) + self.guard[action] > budget:\n'
    '                break\n'
    '            then = perf_counter_ns()\n'
    '            s = s.next(action, get, score)\n'
    '            stage[action] = ms(then)\n'
    '            if action == "dense20" and threshold is not None:\n'
    '                then = perf_counter_ns()\n'
    '                t = ms(start)\n'
    '                paths, base_cost, mean, scale, weights = self.revision.models[s.path]\n'
    '                z = np.clip((np.asarray(s.features(budget, t))[FEATURES] - mean) / scale, -4, 4)\n'
    '                values = weights[:, 0] + weights[:, 1:] @ z - LAMBDA * base_cost / budget\n'
    '                guard_cost = self.suffix_times[s.path]\n'
    '                values[(guard_cost > 0) & (t + guard_cost > budget)] = -np.inf\n'
    '                default = ()\n'
    '                for a in remaining:\n'
    '                    t += self.guard[a]\n'
    '                    if t > budget:\n'
    '                        break\n'
    '                    default += (a,)\n'
    '                best = int(values.argmax())\n'
    '                if paths[best] != default and values[best] - values[paths.index(default)] > threshold:\n'
    '                    remaining = paths[best]\n'
    '                stage["revision"] = ms(then)\n'
    '        duration = ms(start)\n'
    '        return s.rank[:10], duration, s.path, stage\n'
    '\n'
    '\n'
    'def guard_test(engine, seal, sampled):\n'
    '    ds, d = engine.ds, engine.ds.name\n'
    '    file = OUT / f"admission-{d}.json"\n'
    '    assert digest(file) == seal\n'
    '    settings = read(file)\n'
    '    fold = read(FOLLOW / f"frozen-{d}.json")\n'
    '    spec = read(PROJECT / "results" / f"{d}.router.json")\n'
    '    rules = {key: Admission(fold, spec, profile) for key, profile in settings["profiles"].items()}\n'
    '    thresholds = {r["budget"]: r["threshold"] for r in settings["thresholds"]}\n'
    '    reuse = ReuseEngine(engine)\n'
    '    labels = ds.labels("test")\n'
    '    assert set(sampled).isdisjoint(labels)\n'
    '    jobs = [(b, key) for b in settings["budgets"] for key in rules]\n'
    '    for q in ds.source_ids()[0][:3]:\n'
    '        for b, key in jobs:\n'
    '            rules[key].run(reuse, ds.queries[q], b, thresholds[b])\n'
    '    with (OUT / f"guard-{d}.jsonl").open("x") as f:\n'
    '        for q in order(labels, "sensitivity-guard-v1"):\n'
    '            rows = []\n'
    '            sequence = jobs.copy()\n'
    '            random.Random("guard|" + d + "|" + q).shuffle(sequence)\n'
    '            for b, key in sequence:\n'
    '                top, t, path, stages = rules[key].run(reuse, ds.queries[q], b, thresholds[b])\n'
    '                rows.append([b, key, round(t, 6), top, path,\n'
    '                             {k: round(v, 6) for k, v in stages.items()},\n'
    '                             metrics(top, ds, labels[q])["ndcg"]])\n'
    '            f.write(json.dumps([q, rows], separators=(",", ":")) + "\\n")\n'
    '    log("GUARD_DONE", d, len(labels))\n'
    '\n'
    '\n'
    '@torch.inference_mode()\n'
    'def encode(models, texts, cap):\n'
    '    if cap == 192:\n'
    '        return models.encode(texts)\n'
    '    x = models.token(texts, padding=True, truncation=True, max_length=cap, return_tensors="pt").to("cuda")\n'
    '    h = models.enc(**x).last_hidden_state.float()\n'
    '    mean = (h * x.attention_mask[:, :, None]).sum(1) / x.attention_mask.sum(1)[:, None].clamp(min=1)\n'
    '    return F.normalize(mean, dim=1).cpu().numpy()\n'
    '\n'
    '\n'
    '@torch.inference_mode()\n'
    'def rerank(models, text, docs, cap):\n'
    '    if cap == 192:\n'
    '        return models.score([text] * len(docs), docs)\n'
    '    x = models.pairtoken([text] * len(docs), docs, padding=True, truncation=True,\n'
    '                         max_length=cap, return_tensors="pt").to("cuda")\n'
    '    return models.ce(**x).logits.view(-1).float().cpu().tolist()\n'
    '\n'
    '\n'
    'def count_lengths(tokenizer, texts, pair=None):\n'
    '    result = tokenizer(texts, pair, truncation=False, padding=False, return_length=True,\n'
    '                       return_attention_mask=False, return_token_type_ids=False, verbose=False)\n'
    '    return np.asarray(result["length"])\n'
    '\n'
    '\n'
    'def grid_test(engine):\n'
    '    ds, models = engine.ds, engine.models\n'
    '    d = ds.name\n'
    '    # Re-encode every document; both query and document sides use the chosen limit.\n'
    '    assert min(models.enc.config.max_position_embeddings, models.ce.config.max_position_embeddings) >= CAP\n'
    '    index = faiss.IndexFlatIP(engine.ix.d)\n'
    '    start = perf_counter_ns()\n'
    '    for offset in range(0, len(ds.text), 64):\n'
    '        index.add(encode(models, ds.text[offset:offset + 64], CAP))\n'
    '    build_s = ms(start) / 1000\n'
    '    assert index.ntotal == engine.ix.ntotal == len(ds.text)\n'
    '    lengths = np.concatenate([count_lengths(models.token, ds.text[i:i+128])\n'
    '                              for i in range(0, len(ds.text), 128)])\n'
    '    labels = ds.labels("test")\n'
    '    qlengths = count_lengths(models.token, [ds.queries[q] for q in labels])\n'
    '    # Each prefix executes once per query and is charged in full to each suffix.\n'
    '    settings = [(20, 192), (64, 192), (100, 192), (64, CAP)]\n'
    '    def query(q, measure):\n'
    '        query_text = ds.queries[q]\n'
    '        rng = random.Random("grid|" + d + "|" + q)\n'
    '        jobs = settings.copy()\n'
    '        rng.shuffle(jobs)\n'
    '        prefix, results, pair_counts = [], [], []\n'
    '        for k, enc in jobs:\n'
    '            then = perf_counter_ns()\n'
    '            bm = engine.bm.search(query_text, k)[0]\n'
    '            vector = encode(models, [query_text], enc)\n'
    '            ids = (engine.ix if enc == 192 else index).search(vector, k)[1][0].tolist()\n'
    '            candidates = fused(bm, ids)\n'
    '            base_ms = ms(then)\n'
    '            if measure:\n'
    '                rel = labels[q]\n'
    '                hits = sum(rel.get(ds.ids[i], 0) > 0 for i in candidates)\n'
    '                count = sum(x > 0 for x in rel.values())\n'
    '                prefix.append([k, enc, len(candidates), hits, count, round(base_ms, 6)])\n'
    '            suffixes = [(depth, 192) for depth in (0, 8, 24, 50)] if enc == 192 else [(0, 192), (24, 192)]\n'
    '            if k == 64:\n'
    '                suffixes.append((24, CAP))\n'
    '            rng.shuffle(suffixes)\n'
    '            for depth, ce in suffixes:\n'
    '                rank = candidates\n'
    '                t = base_ms\n'
    '                if depth:\n'
    '                    then = perf_counter_ns()\n'
    '                    head = candidates[:depth]\n'
    '                    logits = rerank(models, query_text, [ds.text[i] for i in head], ce)\n'
    '                    rank = [i for _, i in sorted(zip(logits, head), key=lambda x: -x[0])] + candidates[depth:]\n'
    '                    t += ms(then)\n'
    '                if measure:\n'
    '                    top = rank[:10]\n'
    '                    results.append([k, enc, depth, ce, min(depth,len(candidates)), round(t, 6),\n'
    '                                    top, metrics(top, ds, labels[q])["ndcg"]])\n'
    '            if measure and k == 64 and enc == 192:\n'
    '                docs = [ds.text[i] for i in candidates[:24]]\n'
    '                lens = count_lengths(models.pairtoken, [query_text] * len(docs), docs)\n'
    '                pair_counts = [len(lens), int(sum(lens > 192)), int(sum(lens > CAP))]\n'
    '        return [q, prefix, results, pair_counts]\n'
    '    for q in ds.source_ids()[0][:4]:\n'
    '        query(q, False)\n'
    '    with (OUT / f"grid-{d}.jsonl").open("x") as file:\n'
    '        for i, q in enumerate(order(labels, "sensitivity-grid-v1"), 1):\n'
    '            file.write(json.dumps(query(q, True), separators=(",", ":")) + "\\n")\n'
    '            if i % 200 == 0:\n'
    '                log("GRID_PROGRESS", d, i, len(labels))\n'
    '    def counts(values):\n'
    '        return [len(values), int(sum(values > 192)), int(sum(values > CAP))]\n'
    '    save(f"length-{d}.json", dict(documents=counts(lengths), queries=counts(qlengths),\n'
    '                                   documents_reindexed=index.ntotal, build_seconds=build_s))\n'
    '    log("GRID_DONE", d, len(labels), "reindex_s", round(build_s, 1))\n'
    '\n'
    '\n'
    'def main():\n'
    '    OUT.mkdir(exist_ok=False)\n'
    '    faiss.omp_set_num_threads(1)\n'
    '    models = Models()\n'
    '    save("environment.json", dict(gpu=torch.cuda.get_device_name(), models=[ENC, CE],\n'
    '                                  versions={m: version(m) for m in ("numpy", "torch", "faiss-cpu", "transformers")},\n'
    '                                  torch_threads=torch.get_num_threads(), faiss_threads=faiss.omp_get_max_threads(),\n'
    '                                  runner=digest(Path(__file__)), grid_depths=[20,64,100],\n'
    '                                  rerank_depths=[0,8,24,50], limits=[192,CAP]))\n'
    '    engines = {d: Engine(Collection(PROJECT / "data/verified", d), models) for d in DOMAINS}\n'
    '    seals, samples = freeze(engines)\n'
    '    for d in ("fiqa", "scifact", "nfcorpus"):\n'
    '        guard_test(engines[d], seals[d], samples[d])\n'
    '    for d in ("scifact", "nfcorpus", "fiqa"):\n'
    '        grid_test(engines[d])\n'
    '    save("complete.json", {f.name: digest(f) for f in sorted(OUT.iterdir())})\n'
    '    log("SENSITIVITY_COMPLETE", OUT)\n'
    '\n'
    '\n'
    'if __name__ == "__main__":\n'
    '    main()\n'
)
runner=work/'sensitivity_run.py'
runner.write_bytes(runner_source.encode('utf-8'))
assert hashlib.sha256(runner.read_bytes()).hexdigest()=='1f6de733c5d58902e42d0ba9eaeb589920783fb66c488deb6f39dd91e7171cd4'

In [ ]:
with subprocess.Popen([sys.executable,'-u',str(runner)], cwd=work,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as proc:
    for line in proc.stdout:
        print(line.rstrip(),flush=True)
    assert proc.wait()==0

In [ ]:
out=work/'sensitivity-output'
manifest=json.loads((out/'complete.json').read_text())
for name,digest in manifest.items():
    assert hashlib.sha256((out/name).read_bytes()).hexdigest()==digest
archive=work/'observations.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED,compresslevel=9) as z:
    for f in sorted(out.iterdir()): z.write(f,f.name)
print('Verified',len(manifest),'files. Archive SHA-256',hashlib.sha256(archive.read_bytes()).hexdigest())
files.download(str(archive))
